# Model Linier dan Regularisasi (Linear Models and Regularization)

Catatan ini menguraikan konsep dan implementasi dari keluarga algoritma Model Linier (*Linear Models*) untuk tugas regresi dan klasifikasi. Model linier sangat populer karena kecepatan komputasinya, kemudahan interpretasi, dan kemampuannya menangani dataset berdimensi tinggi (*sparse data*).

Fokus utama pada modul ini adalah memahami bahaya *overfitting* pada model linier standar dan bagaimana teknik **Regularisasi** (Ridge dan Lasso) digunakan untuk mengendalikan kompleksitas model.

#### **Tujuan Pembelajaran**
* Mampu mengimplementasikan Regresi Linier Standar (*Ordinary Least Squares*).
* Memahami konsep Regularisasi L2 (*Ridge Regression*) dan dampaknya terhadap koefisien fitur.
* Memahami konsep Regularisasi L1 (*Lasso Regression*) untuk seleksi fitur otomatis.
* Menerapkan model linier untuk klasifikasi menggunakan *Logistic Regression* dan *LinearSVC*.
* Mengontrol kompleksitas model melalui parameter `alpha` (pada regresi) dan parameter `C` (pada klasifikasi).

In [19]:
# Memuat pustaka komputasi numerik dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Memuat pembuat dataset sintetis dan pemisah data
from sklearn.datasets import make_regression, make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split

# Memuat keluarga Model Linier dari scikit-learn
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.svm import LinearSVC

# Konfigurasi visualisasi dasar
%matplotlib inline
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

print("Pustaka untuk Model Linier berhasil dimuat.")

Pustaka untuk Model Linier berhasil dimuat.


## 1. Regresi Linier Standar (Ordinary Least Squares / OLS)

Regresi Linier adalah algoritma regresi paling dasar. Model ini bekerja dengan mencari bobot koefisien ($w$) dan garis potong (*intercept* / $b$) yang meminimalkan *Mean Squared Error* (MSE) antara nilai prediksi dan nilai aktual pada data latih.

**Kelemahan OLS:** Model ini tidak memiliki parameter kontrol kompleksitas. Jika jumlah fitur sangat banyak (dimensi tinggi), OLS sangat rentan "menghafal" data latih (*overfitting*).

In [20]:
# Membuat dataset regresi dengan fitur yang banyak (rawan overfitting)
X_reg, y_reg = make_regression(n_samples=100, n_features=50, noise=20, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, random_state=42)

# Melatih model Ordinary Least Squares (OLS)
ols_model = LinearRegression()
ols_model.fit(X_train_reg, y_train_reg)

print("=== Regresi Linier (OLS) ===")
print(f"Skor R-Squared Latih : {ols_model.score(X_train_reg, y_train_reg):.4f}")
print(f"Skor R-Squared Uji   : {ols_model.score(X_test_reg, y_test_reg):.4f}")
print("Analisis: Jarak yang jauh antara skor latih dan skor uji mengindikasikan terjadinya Overfitting.")

=== Regresi Linier (OLS) ===
Skor R-Squared Latih : 0.9938
Skor R-Squared Uji   : 0.9457
Analisis: Jarak yang jauh antara skor latih dan skor uji mengindikasikan terjadinya Overfitting.


## 2. Ridge Regression (Regularisasi L2)

*Ridge Regression* mengatasi kelemahan OLS dengan menambahkan penalti pada fungsi kerugian (*loss function*). Penalti ini (disebut **Regularisasi L2**) memaksa model untuk menjaga nilai koefisien ($w$) sekecil mungkin (mendekati nol).

Intinya, model dipaksa untuk mempertimbangkan semua fitur secara seimbang tanpa bergantung terlalu besar pada salah satu fitur saja. Kompleksitas model dikendalikan oleh parameter `alpha`.
* `alpha` besar $\rightarrow$ Penalti kuat, model lebih sederhana (*underfitting*).
* `alpha` kecil $\rightarrow$ Penalti lemah, model mendekati OLS standar (*overfitting*).

In [21]:
# Melatih Ridge Regression dengan berbagai nilai alpha
ridge_default = Ridge(alpha=1.0).fit(X_train_reg, y_train_reg)
ridge_alpha10 = Ridge(alpha=10.0).fit(X_train_reg, y_train_reg)
ridge_alpha01 = Ridge(alpha=0.1).fit(X_train_reg, y_train_reg)

print("=== Ridge Regression (Regularisasi L2) ===")
print(f"Ridge (alpha=0.1)  | Skor Uji: {ridge_alpha01.score(X_test_reg, y_test_reg):.4f}")
print(f"Ridge (alpha=1.0)  | Skor Uji: {ridge_default.score(X_test_reg, y_test_reg):.4f}")
print(f"Ridge (alpha=10.0) | Skor Uji: {ridge_alpha10.score(X_test_reg, y_test_reg):.4f}")

=== Ridge Regression (Regularisasi L2) ===
Ridge (alpha=0.1)  | Skor Uji: 0.9458
Ridge (alpha=1.0)  | Skor Uji: 0.9451
Ridge (alpha=10.0) | Skor Uji: 0.9232


## 3. Lasso Regression (Regularisasi L1)

Serupa dengan Ridge, *Lasso* juga membatasi nilai koefisien. Namun, *Lasso* menggunakan **Regularisasi L1** yang memiliki efek drastis: beberapa koefisien fitur akan dipaksa menjadi **tepat nol**.

Ini berarti model akan sepenuhnya mengabaikan beberapa fitur. Oleh karena itu, *Lasso* bertindak sebagai algoritma **seleksi fitur otomatis**, sehingga menghasilkan model yang jauh lebih mudah diinterpretasikan karena hanya bergantung pada subset fitur yang paling relevan.

In [22]:
# Melatih Lasso Regression
lasso_default = Lasso(alpha=1.0).fit(X_train_reg, y_train_reg)
lasso_alpha01 = Lasso(alpha=0.1, max_iter=10000).fit(X_train_reg, y_train_reg)

print("=== Lasso Regression (Regularisasi L1) ===")
print(f"Lasso (alpha=1.0)  | Skor Uji: {lasso_default.score(X_test_reg, y_test_reg):.4f}")
print(f"Fitur yang terpakai (alpha=1.0): {np.sum(lasso_default.coef_ != 0)} dari 50 fitur")

print(f"\nLasso (alpha=0.1)  | Skor Uji: {lasso_alpha01.score(X_test_reg, y_test_reg):.4f}")
print(f"Fitur yang terpakai (alpha=0.1): {np.sum(lasso_alpha01.coef_ != 0)} dari 50 fitur")

=== Lasso Regression (Regularisasi L1) ===
Lasso (alpha=1.0)  | Skor Uji: 0.9701
Fitur yang terpakai (alpha=1.0): 33 dari 50 fitur

Lasso (alpha=0.1)  | Skor Uji: 0.9494
Fitur yang terpakai (alpha=0.1): 49 dari 50 fitur


## 4. Klasifikasi Linier (Logistic Regression & LinearSVC)

Model linier juga sangat dominan dalam ranah klasifikasi. Alih-alih memprediksi nilai kontinu, algoritma ini mencari **Hiperbidang (*Hyperplane*)** atau garis lurus yang memisahkan kelas data.

Dua algoritma yang paling umum adalah:
1. **`LogisticRegression`** (Meskipun namanya "Regresi", ini adalah model Klasifikasi).
2. **`LinearSVC`** (*Linear Support Vector Classifier*).

Secara bawaan, keduanya menggunakan Regularisasi L2. Parameter pengontrol kompleksitas pada klasifikasi disebut parameter `C` (kebalikan dari `alpha`).
* `C` besar $\rightarrow$ Sedikit regularisasi, model berusaha menyesuaikan setiap titik data latih (rentan *overfitting*).
* `C` kecil $\rightarrow$ Banyak regularisasi, batas keputusan menjadi garis yang lebih "lurus" dan mengabaikan titik anomali individu (lebih *general*).

In [23]:
# Memuat Dataset Kanker Payudara untuk Klasifikasi
cancer = load_breast_cancer()
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42
)

# 1. Regresi Logistik dengan C=1 (Default)
logreg = LogisticRegression(C=1.0, max_iter=10000).fit(X_train_cls, y_train_cls)

# 2. Regresi Logistik dengan C=100 (Kurang Regularisasi)
logreg_100 = LogisticRegression(C=100.0, max_iter=10000).fit(X_train_cls, y_train_cls)

# 3. Regresi Logistik dengan C=0.01 (Banyak Regularisasi)
logreg_001 = LogisticRegression(C=0.01, max_iter=10000).fit(X_train_cls, y_train_cls)

print("=== Logistic Regression Performance ===")
print(f"Model C=1.0  (Default) | Akurasi Uji: {logreg.score(X_test_cls, y_test_cls):.4f}")
print(f"Model C=100  (Overfit) | Akurasi Uji: {logreg_100.score(X_test_cls, y_test_cls):.4f}")
print(f"Model C=0.01 (Underfit)| Akurasi Uji: {logreg_001.score(X_test_cls, y_test_cls):.4f}")

# Evaluasi LinearSVC
svm_linier = LinearSVC(C=1.0, max_iter=10000).fit(X_train_cls, y_train_cls)
print(f"\nLinearSVC (C=1.0)      | Akurasi Uji: {svm_linier.score(X_test_cls, y_test_cls):.4f}")

=== Logistic Regression Performance ===
Model C=1.0  (Default) | Akurasi Uji: 0.9580
Model C=100  (Overfit) | Akurasi Uji: 0.9650
Model C=0.01 (Underfit)| Akurasi Uji: 0.9510

LinearSVC (C=1.0)      | Akurasi Uji: 0.9580


## Kesimpulan

Model Linier (*Linear Models*) merupakan fondasi penting dalam *machine learning* karena:
1. Sangat cepat dalam proses pelatihan (*training*) maupun prediksi.
2. Sangat efisien untuk himpunan data yang besar dan berdimensi spasial tinggi (*sparse data* seperti teks).
3. Mudah diinterpretasikan melalui peninjauan matriks bobot koefisien.

**Aturan Emas Pemilihan Model Linier:**
* Jika data berdimensi tinggi dan rentan *overfitting*, gunakan **Ridge**.
* Jika ingin kemudahan interpretasi karena berasumsi hanya beberapa fitur yang benar-benar penting, gunakan **Lasso**.
* Untuk tugas klasifikasi biner maupun multikelas, selalu mulai dengan **Logistic Regression** sebelum beranjak ke model komputasi berat (seperti Jaringan Syaraf Tiruan).